# Smart Onboarding Assistant — A/B Test Analysis
**Mobile productivity app "Streamline" · 7,500 users · Primary metric: 7-day retention**

**Business context.** 7-day retention has been flat at ~36% for three quarters. Product built a *Smart Onboarding Assistant* — a guided first-session setup — hypothesizing that faster time-to-value will increase D7 retention. Each 1pp of D7 retention is worth ≈ $400K/year in downstream subscription revenue. Leadership will ship only if the test shows a statistically significant lift of **≥ 3pp** (the MDE that pays for the ~2 FTE maintenance cost).

**Hypotheses.**
- H₀: retention(Treatment) = retention(Control)
- H₁: retention(Treatment) ≠ retention(Control), α = 0.05 (two-sided)

**Decision rule (pre-registered).** Ship if p < 0.05 AND observed lift ≥ 3pp. Guardrails: session length and screens viewed must not degrade.

**Analysis plan.** (1) Design & power review → (2) Data audit → (3) Experiment health checks (SRM, covariate balance) → (4) Primary metric: two-proportion z-test + CI + effect size → (5) Secondary metrics: Mann-Whitney U → (6) Segment analysis with multiple-comparison correction → (7) Adoption deep-dive → (8) Business impact & recommendation.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import (proportions_ztest, proportion_confint,
                                          confint_proportions_2indep, proportion_effectsize)
from statsmodels.stats.power import NormalIndPower
import matplotlib.pyplot as plt

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
COL_C, COL_T = '#8C8C8C', '#2E86AB'

## 1 · Design review & power analysis
Before touching results, a senior analyst re-derives the required sample size. Baseline retention ≈ 36.5%; MDE = +3pp; α = 0.05; power = 80%.

In [ ]:
power = NormalIndPower()
es_mde = proportion_effectsize(0.395, 0.365)          # Cohen's h for 36.5% -> 39.5%
n_required = power.solve_power(effect_size=es_mde, alpha=0.05, power=0.80,
                               alternative='two-sided')
print(f"Required per arm for 3pp MDE @ 80% power: {n_required:,.0f}")

actual_power = power.solve_power(effect_size=es_mde, nobs1=3750, alpha=0.05,
                                 alternative='two-sided')
print(f"Actual power of this design (n≈3,750/arm) for a true 3pp effect: {actual_power:.1%}")

> **Finding — flag it, own it.** The design needed ~4,108 users/arm for 80% power at the 3pp MDE; it enrolled ~3,750/arm, giving ~76% power. Slightly underpowered — meaning a true 3pp effect had a ~24% chance of going undetected. It does **not** invalidate a significant result; it raises the risk of a false *negative* and (mildly) of effect-size exaggeration in significant results ("winner's curse"). We report it transparently and lean on the confidence interval, not the point estimate, for the business case.

## 2 · Data audit

In [ ]:
df = pd.read_csv('mobile_app_ab_test_dataset.csv')
df['retained'] = (df['Retained_7_Days'] == 'Yes').astype(int)
df['feature']  = (df['Feature_Used']  == 'Yes').astype(int)

print(df.shape)
print(df.isnull().sum().sum(), "nulls |", df['User_ID'].duplicated().sum(), "duplicate user IDs")
df.head()

In [ ]:
# Distribution sanity: engagement metrics are heavily right-skewed and session
# length is hard-capped at 7200s by the logging system (visible as max in BOTH arms).
print(df[['Session_Length','Screens_Viewed']].describe().round(1))
print("\nSkewness:", stats.skew(df.Session_Length).round(2), "(session),",
      stats.skew(df.Screens_Viewed).round(2), "(screens)")
print("Max session per group:\n", df.groupby('Group')['Session_Length'].max())

**Audit verdict:** 7,500 rows, one per user, zero nulls, zero duplicate IDs. Engagement metrics are right-skewed (skew ≈ 3.3–3.6) with a hard cap at 7,200s → we will use **medians and non-parametric tests** for those, never trimming "outliers" (they're real heavy users, and trimming an A/B test post-hoc is how you manufacture false positives).

## 3 · Experiment health checks
An A/B test result is only as trustworthy as its randomization. Two checks before any metric is read:
1. **Sample Ratio Mismatch (SRM)** — does the observed split match the intended 50/50?
2. **Covariate balance** — are device and app version distributed identically across arms?

In [ ]:
n_c = (df.Group=='Control').sum(); n_t = (df.Group=='Treatment').sum()
srm_stat, srm_p = stats.chisquare([n_c, n_t], f_exp=[len(df)/2, len(df)/2])
print(f"SRM: Control={n_c:,}  Treatment={n_t:,}  chi2={srm_stat:.3f}  p={srm_p:.4f}")

dev_p = stats.chi2_contingency(pd.crosstab(df.Group, df.Device)).pvalue
ver_p = stats.chi2_contingency(pd.crosstab(df.Group, df.App_Version)).pvalue
print(f"Balance — Device: p={dev_p:.4f} | App_Version: p={ver_p:.4f}")

leak = df[(df.Group=='Control') & (df.feature==1)]
print(f"Contamination: {len(leak)} Control users ({len(leak)/n_c:.2%}) logged feature usage")

**Health verdict: PASS.** SRM p = 0.41 (no assignment bug), covariates balanced (p = 0.76, 0.92). Contamination is 1.02% of Control — trivial; if anything it biases the measured lift slightly *toward zero* (conservative). We analyze **intent-to-treat**: every user stays in their assigned arm regardless of whether they used the feature.

## 4 · Primary metric — 7-day retention (two-proportion z-test)

In [ ]:
t = df[df.Group=='Treatment']; c = df[df.Group=='Control']
successes = np.array([t.retained.sum(), c.retained.sum()])
nobs      = np.array([len(t), len(c)])

z_stat, p_value = proportions_ztest(successes, nobs, alternative='two-sided')
p_t, p_c = successes / nobs
lift_abs = p_t - p_c
ci_lo, ci_hi = confint_proportions_2indep(successes[0], nobs[0],
                                          successes[1], nobs[1], method='wald')
cohens_h = proportion_effectsize(p_t, p_c)

print(f"Control:   {p_c:.4f}  ({successes[1]:,}/{nobs[1]:,})")
print(f"Treatment: {p_t:.4f}  ({successes[0]:,}/{nobs[0]:,})")
print(f"Absolute lift: {lift_abs*100:+.2f}pp | Relative: {lift_abs/p_c:+.1%}")
print(f"z = {z_stat:.4f}, p = {p_value:.6f}")
print(f"95% CI for lift: [{ci_lo*100:.2f}, {ci_hi*100:.2f}]pp")
print(f"Cohen's h = {cohens_h:.4f} (small in stats terms — normal for retention metrics)")

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
for i,(g,d,col) in enumerate([('Control',c,COL_C),('Treatment',t,COL_T)]):
    p = d.retained.mean()
    lo, hi = proportion_confint(d.retained.sum(), len(d), method='wilson')
    ax.bar(i, p*100, color=col, width=0.55)
    ax.errorbar(i, p*100, yerr=[[(p-lo)*100],[(hi-p)*100]], color='black',
                capsize=6, fmt='none', lw=1.5)
    ax.text(i, p*100+1.8, f"{p*100:.1f}%", ha='center', fontsize=13, fontweight='bold')
ax.set_xticks([0,1])
ax.set_xticklabels(['Control\n(standard onboarding)','Treatment\n(Smart Onboarding Assistant)'])
ax.set_ylabel('7-Day Retention Rate (%)'); ax.set_ylim(0,50)
ax.set_title('Primary Metric: 7-Day Retention\n+4.11pp lift (p=0.0003, 95% CI [1.91, 6.31]pp)')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

**Read.** Treatment retention 40.7% vs Control 36.5% → **+4.11pp (+11.3% relative), p = 0.0003**. If the feature truly did nothing, we'd see a gap this large in ~3 of 10,000 experiments. The CI says the true lift is plausibly anywhere from **+1.9pp to +6.3pp** — the point estimate clears the 3pp ship bar, but the lower bound doesn't. That tension *is* the executive conversation (Section 8).

## 5 · Secondary metrics — engagement guardrails
Right-skewed + capped ⇒ Mann-Whitney U (compares distributions without normality assumptions); Welch's t reported as a robustness check.

In [ ]:
for col in ['Session_Length','Screens_Viewed']:
    mw = stats.mannwhitneyu(t[col], c[col], alternative='two-sided')
    tt = stats.ttest_ind(t[col], c[col], equal_var=False)
    print(f"{col:15s}  Ctrl med={c[col].median():6.0f}  Trt med={t[col].median():6.0f}"
          f"  | Mann-Whitney p={mw.pvalue:.5f} | Welch p={tt.pvalue:.5f}")

**Read.** Both guardrails moved *up*, not just "didn't degrade": median session +25s (420→445, p=0.005), median screens +2 (12→14, p<0.0001). Engagement corroborates the retention story — users given the Assistant do more in the app. Consistent metrics moving together is what a real effect looks like; a retention lift with flat engagement would have made me suspicious.

## 6 · Segment analysis — is the effect consistent?
Seven segment tests ⇒ Bonferroni-corrected α = 0.05/7 ≈ 0.0071. **Rule I've enforced for 20 years:** segments *describe* the effect; they never *rescue* a failed test or crown surprise winners without a follow-up experiment.

In [ ]:
segments = ([('Device', v) for v in ['iOS','Android']] +
            [('App_Version', v) for v in sorted(df.App_Version.unique())])
rows = []
for col, val in segments:
    s = df[df[col]==val]
    st_, sc_ = s[s.Group=='Treatment'], s[s.Group=='Control']
    suc = np.array([st_.retained.sum(), sc_.retained.sum()])
    nb  = np.array([len(st_), len(sc_)])
    z_, p_ = proportions_ztest(suc, nb)
    rows.append({'segment': f"{col}={val}", 'n': nb.sum(),
                 'lift_pp': (suc[0]/nb[0]-suc[1]/nb[1])*100, 'p': p_,
                 'sig_after_bonferroni': p_ < 0.05/len(segments)})
pd.DataFrame(rows).round(4)

**Read.** Direction is positive in all 7 segments — the effect is *consistent*, which matters more than segment-level significance (segments are underpowered by construction). iOS +3.9pp and Android +4.4pp are statistically indistinguishable from each other. Version 3.3.0's +11.3pp survives Bonferroni but at n≈1,200 I treat it as noise-until-replicated, not a finding. Junior analysts build stories on cells like that; seniors note it and move on.

## 7 · Adoption deep-dive — the "what's next" insight

In [ ]:
adoption = t.feature.mean()
print(f"Feature adoption within Treatment: {adoption:.1%}")
print(f"Retention — Treatment adopters:     {t[t.feature==1].retained.mean():.1%}  (n={t.feature.sum():,})")
print(f"Retention — Treatment non-adopters: {t[t.feature==0].retained.mean():.1%}")
print(f"Retention — Control (baseline):     {c.retained.mean():.1%}")

**Read — with the causal caveat stated out loud.** Only **37.8%** of Treatment users engaged with the Assistant. Adopters retained at **50.5%** vs 34.7% for non-adopters. This comparison is **descriptive, not causal** — motivated users self-select into both adoption and retention. The *causal* number remains the ITT +4.11pp. But the pattern (non-adopters ≈ Control baseline) is exactly what a genuine feature effect looks like, and it hands product the roadmap: **the ceiling on this feature is adoption.** If discoverability work moved adoption from 38% toward 60%, even a diluted per-adopter effect implies materially more total lift — that's the follow-up experiment, not a promise.

## 8 · Business impact & recommendation

| Question | Answer |
|---|---|
| Statistically significant? | **Yes** — p = 0.0003, z = 3.66 |
| Clears the 3pp ship bar? | **Point estimate yes (4.11pp); CI lower bound (1.91pp) does not** |
| Guardrails healthy? | **Yes** — both engagement metrics improved |
| Randomization trustworthy? | **Yes** — SRM and balance checks pass |
| Annualized value | **$0.76M (conservative, CI lower bound) to $2.5M; point estimate ≈ $1.6M** |

### Recommendation: **SHIP** — with two conditions
1. **Ship to 100%** of new users. Even the worst plausible case ($0.76M/yr) covers the ~2-FTE maintenance cost; expected value is ~$1.6M/yr.
2. **Fund the adoption follow-up.** 62% of exposed users never touched the Assistant. Next experiment: discoverability/trigger variants aimed at raising adoption from 38% → 60%.

### Honest limitations (say these before the VP asks)
- Test ran at ~76% power for the 3pp MDE (needed ~4.1K/arm, had ~3.75K) — a plausible true effect below 3pp can't be ruled out; hence the CI-based conservative case.
- D7 retention is an early proxy; recommend a 30/90-day holdback (e.g., 5% of users) to confirm long-run effect and watch for novelty decay.
- 1% Control contamination — trivial, direction conservative.
- Segment findings are descriptive; v3.3.0 outlier not actionable without replication.